# 03 · Líneas base y modelos candidatos

Todos los modelos usan **la misma preparación** (`src/pipeline.py`), **la misma validación** (5 pliegues temporales de una semana cada uno, con ventana expansiva) y **la misma métrica** (WAPE). Cada corrida queda registrada en MLflow con datos, parámetros, métricas, tags y el modelo entrenado.

| Fase | Corridas |
|---|---|
| Línea base | Dummy (mediana) y la regla de negocio del PSet 1 (promedio de 4 semanas) |
| Candidatos | Ridge, RandomForest, HistGradientBoosting, LightGBM, MLP, Chronos-2 y un LLM por prompting |

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")   # avisos de esquema de MLflow
import os, logging
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "1")
logging.getLogger("mlflow").setLevel(logging.ERROR)   # avisos repetidos de serialización/entorno
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor

import src.pipeline as P
import src.tracking as T
from src.datos import cargar_obt
from src.modelos import Chronos2Pronostico, LLMPronostico
from src import estilo
estilo.aplicar()

print("MLflow:", T.configurar(), "· experimento:", T.EXPERIMENTO, "· commit:", T.commit_git())
snap = cargar_obt()
div = P.dividir(P.agregar_features(snap.datos))
pliegues = P.pliegues_temporales(div.train)
for i, (a, b) in enumerate(pliegues):
    print(f"pliegue {i}: entrena {len(a)} filas hasta {div.train.hour_date.iloc[a].max():%d-%m} · valida {len(b)} filas "
          f"({div.train.hour_date.iloc[b].min():%d-%m} a {div.train.hour_date.iloc[b].max():%d-%m})")

In [ ]:
resultados = {}

def correr(nombre, modelo, fase, familia):
    res = P.evaluar(modelo, div.train, pliegues)
    est = modelo.named_steps["modelo"] if hasattr(modelo, "named_steps") else modelo
    params = est.get_params()
    lr = params.get("learning_rate_init", params.get("learning_rate"))   # MLP: learning_rate_init
    run_id = T.registrar(nombre, modelo, res, fase=fase, familia=familia, snapshot=snap, train=div.train)
    resultados[nombre] = {"fase": fase, "familia": familia, **res.metricas,
                          "learning_rate": lr if isinstance(lr, (int, float)) else np.nan,
                          "run_id": run_id, "oof": res.oof, "pliegues": res.wape_pliegues}
    m = res.metricas
    print(f"{nombre:<22} train {m['train_wape']:.3f} · CV {m['cv_wape_mean']:.3f} ± {m['cv_wape_std']:.3f} · "
          f"costo ${m['cv_costo_negocio']:.0f} · {m['fit_seconds']:.2f} s · run {run_id[:8]}")

## Líneas base

In [ ]:
correr("dummy_mediana", DummyRegressor(strategy="median"), "baseline", "ingenua")
correr("regla_prom_4_semanas", P.ReglaPromedio4Semanas(), "baseline", "negocio")

## Candidatos con hiperparámetros por defecto

In [ ]:
candidatos = {
    "ridge": (P.construir_pipeline(Ridge()), "lineal_regularizado"),
    "random_forest": (P.construir_pipeline(RandomForestRegressor(random_state=P.SEMILLA, n_jobs=-1)), "bagging"),
    "hist_gradient_boosting": (P.construir_pipeline(HistGradientBoostingRegressor(random_state=P.SEMILLA)), "boosting"),
    "lightgbm": (P.construir_pipeline(LGBMRegressor(random_state=P.SEMILLA, verbose=-1)), "boosting"),
    "mlp": (P.construir_pipeline(MLPRegressor(random_state=P.SEMILLA)), "red_neuronal"),
}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # la MLP por defecto no converge en 200 iteraciones: se ve en el tuning
    for nombre, (modelo, familia) in candidatos.items():
        correr(nombre, modelo, "candidato", familia)

### State of the art: Chronos-2 (modelo fundacional de series de tiempo)

Chronos-2 (Amazon, 2025) se usa sin entrenamiento (*zero-shot*): recibe la serie de ventas por hora abierta y, como covariables futuras conocidas, la hora, el día, los feriados y la ventana de pago. No pasa por el `ColumnTransformer` porque consume la serie y no la matriz de features. Sí usa los mismos pliegues y la misma métrica. Corre en un proceso aparte (`src/chronos_worker.py`), porque el OpenMP de torch choca con el de LightGBM dentro del mismo kernel.

In [ ]:
correr("chronos2_zero_shot", Chronos2Pronostico(), "candidato", "fundacional")

### LLM por prompting (Claude Haiku 5.5)

El problema **sí** se puede plantear como prompting: se entregan como tabla CSV las 4 semanas previas (fecha, día, hora, feriado, ventas) y se piden las ventas de cada hora de la semana siguiente, en JSON validado. Se usa un modelo pequeño a propósito: la tarea es aritmética de patrones, no razonamiento abierto, y el costo debe ser marginal.

Las respuestas de un LLM no son deterministas. Para que la corrida sea reproducible, cada prompt y su respuesta se guardan en `docs/llm/`: al re-ejecutar el notebook se leen de ahí, sin llamar a la API.

In [ ]:
correr("llm_claude_haiku", LLMPronostico(), "candidato", "llm")
import json
uso = [json.loads(p.read_text())["uso"] for p in (RAIZ / "docs" / "llm").glob("*.json")]
tok_in, tok_out = sum(u["input_tokens"] for u in uso), sum(u["output_tokens"] for u in uso)
print(f"{len(uso)} llamadas · {tok_in:,} tokens de entrada · {tok_out:,} de salida · ≈ ${tok_in/1e6*0.10 + tok_out/1e6*0.50:.3f}")

## Comparación

In [ ]:
tabla = (pd.DataFrame(resultados).T
         [["fase", "familia", "train_wape", "cv_wape_mean", "cv_wape_std", "brecha_wape", "learning_rate",
           "fit_seconds", "cv_mae_mean", "cv_costo_negocio", "cv_recall_sobrecarga", "run_id"]]
         .astype({c: float for c in ["train_wape", "cv_wape_mean", "cv_wape_std", "brecha_wape", "learning_rate",
                                       "fit_seconds", "cv_mae_mean", "cv_costo_negocio", "cv_recall_sobrecarga"]}))
tabla = tabla.sort_values("cv_wape_mean")
regla, dummy = tabla.loc["regla_prom_4_semanas", "cv_wape_mean"], tabla.loc["dummy_mediana", "cv_wape_mean"]
tabla["supera_lineas_base"] = (tabla.cv_wape_mean < min(regla, dummy)) & (tabla.fase == "candidato")
tabla["mejora_vs_regla_en_desv"] = (regla - tabla.cv_wape_mean) / tabla.cv_wape_std
(RAIZ / "docs").mkdir(exist_ok=True)
tabla.drop(columns="run_id").round(4).to_csv(RAIZ / "docs" / "resultados_candidatos.csv")
tabla.drop(columns="run_id").round(3)

In [ ]:
orden = tabla.index.tolist()
col = {"ingenua": estilo.GRIS, "negocio": estilo.AMBAR}
colores = [col.get(tabla.loc[n, "familia"], estilo.INDIGO) for n in orden]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[1.6, 1]))
y = np.arange(len(orden))
ax[0].barh(y, tabla.cv_wape_mean, xerr=tabla.cv_wape_std, color=colores, height=0.6,
           error_kw=dict(ecolor=estilo.TINTA_2, lw=1, capsize=3), label="CV (media ± desv.)")
ax[0].scatter(tabla.train_wape, y, marker="D", s=28, color=estilo.TEAL, zorder=3, label="entrenamiento")
ax[0].axvline(regla, color=estilo.AMBAR, lw=1, ls="--")
ax[0].set_yticks(y, orden); ax[0].invert_yaxis()
ax[0].set(xlabel="WAPE (menor es mejor)", title="Entrenamiento vs. validación cruzada")
ax[0].legend(loc="lower right", fontsize=8)
ax[1].barh(y, tabla.cv_costo_negocio, color=colores, height=0.6)
ax[1].set_yticks(y, [""] * len(orden)); ax[1].invert_yaxis()
ax[1].set(xlabel="costo en los 5 pliegues (USD)", title="Costo de negocio")
for i, v in enumerate(tabla.cv_costo_negocio):
    ax[1].text(v + 4, i, f"${v:.0f}", va="center", fontsize=8)
fig.tight_layout(); estilo.guardar(fig, "candidatos_comparacion"); plt.show()

In [ ]:
# Desempeño por pliegue: ¿el orden entre modelos es estable o depende de la semana?
pl = pd.DataFrame({n: resultados[n]["pliegues"] for n in orden},
                  index=[f"{div.train.hour_date.iloc[b].min():%d-%m}" for _, b in pliegues])
pl.round(3)

## Diagnóstico

**¿Superan a las dos líneas base?** Todos superan a la línea base ingenua. Frente a la regla de negocio (WAPE 0,411):

- **Ridge (0,379) y MLP (0,392) la superan.** Ridge le gana en los **5 de 5 pliegues**. Aun así, su ventaja media (0,032) es apenas menor que una desviación entre pliegues (0,035): es una mejora consistente, pero pequeña.
- **RandomForest (0,410), Chronos-2 (0,409) y el LLM (0,411) empatan con la regla.** **HistGradientBoosting (0,421) y LightGBM (0,416) no la superan.** *Hallazgo:* el promedio de 4 semanas ya entra como variable, así que un modelo solo aporta si aprende algo más que ese promedio. Con unas 640 filas por pliegue, los árboles por defecto no lo logran.

**Sobreajuste.** Ridge tiene una brecha entre entrenamiento y CV de 0,036 (sana) y la MLP de 0,079 (leve). Los árboles por defecto memorizan: RandomForest llega a 0,14 en entrenamiento y 0,41 en CV (brecha 0,27), y los dos boosting a 0,21. Es la consecuencia esperada del hallazgo H5 (pocas filas y variables redundantes). En el tuning, el espacio de RandomForest limita la profundidad y el tamaño de las hojas.

**Tiempo.** Todos entrenan en menos de un segundo. Chronos-2 no entrena, pero cada predicción tarda unos 3 s en CPU. El LLM tarda unos 15 s por semana, cuesta alrededor de un centavo por las 5 semanas y no es determinista (por eso se guardan sus respuestas).

**Costo de negocio.** La regla tiene el costo más bajo ($214), por debajo de todos los modelos. Los modelos aplanan los picos: predicen cerca de la media en las horas extremas y, con el umbral fijo de 8, dejan pasar horas sobrecargadas. Esto motiva calibrar el umbral de decisión según el costo (notebook 05).

**El LLM.** Con un prompt simple (la tabla de 4 semanas) iguala a la regla. Lo más probable es que haga algo muy parecido a promediar semanas anteriores. No aporta explicabilidad, depende de un servicio externo y su salida varía entre llamadas: no es candidato a producción.

**Los 3 que pasan al tuning:** Ridge, MLP y RandomForest, los mejores *entrenables*. Chronos-2 queda fuera por ser *zero-shot*, aunque su WAPE (0,409) sea apenas mejor que el de RandomForest (0,410).